# 02 · Remaining-useful-life benchmark on NASA C-MAPSS FD001

**Setup.** 100 training engines run to failure, 100 test engines truncated before failure with the true RUL supplied by NASA. Features per (unit, cycle): raw value, 30-cycle rolling mean, rolling std and window slope for the 14 informative sensors (the 7 constant sensors are dropped). Labels capped at 125 cycles: an engine 300 cycles from failure and one 200 cycles away are both simply healthy, and letting the model chase the far tail wastes capacity on the region nobody acts on.

**Metrics.** RMSE and the PHM08 asymmetric score, which penalises late predictions (`exp(d/10)-1`) far harder than early ones (`exp(-d/13)-1`). In maintenance, optimism is the expensive error, so the score is the honest headline.

**Models.** LightGBM (L1 objective, 1200 trees), a two-layer GRU on the same 30-cycle windows, and a constant baseline that predicts the training-set mean capped RUL for every engine. The GRU is trained in a subprocess: torch and LightGBM ship separate OpenMP runtimes and deadlock in one process on macOS.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
m = json.load(open('../data/metrics/ajal_rul.json'))
print(f"{m['dataset']} · {m['n_train_units']} train engines · {m['n_test_units']} test engines · RUL cap {m['rul_cap']}")
table = pd.DataFrame(m['models']).T[['version','rmse','nasa_score','fit_seconds']]
table['rmse'] = table.rmse.astype(float).round(2); table['nasa_score'] = table.nasa_score.astype(float).round(1); table['fit_seconds'] = table.fit_seconds.astype(float).round(1)
table

FD001 · 100 train engines · 100 test engines · RUL cap 125


,version,rmse,nasa_score,fit_seconds
lgbm,ajal-rul/lgbm-l1-w30-cap125/2026-09,15.36,463.9,17.4
gru,ajal-rul/gru-h64-seq30/2026-09,16.92,954.6,31.3
last_cycle_constant,baseline/train-mean-capped-rul,41.94,33354.5,0.0


In [2]:
pred = pd.DataFrame(m['predictions'])
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharex=True, sharey=True)
for ax, col, colour in zip(axes, ['lgbm', 'gru', 'baseline'], ['#E8A317', '#8FA8B5', '#C4342B']):
    ax.scatter(pred.true_rul, pred[col], s=14, color=colour, alpha=0.8)
    ax.plot([0, 125], [0, 125], color='#2F4E5F', lw=1)
    d = pred[col] - pred.true_rul
    ax.set_title(f"{col}  RMSE {np.sqrt((d**2).mean()):.1f}")
    ax.set_xlabel('true RUL (capped)')
axes[0].set_ylabel('predicted RUL')
fig.tight_layout(); fig.savefig('../docs/figures/rul_pred_vs_true.png'); plt.close(fig)
# Where does the error sit? Late predictions are the costly ones.
late = (pred.lgbm > pred.true_rul)
print(f"LightGBM: {late.mean():.0%} of test engines predicted late; mean lateness among them {(pred.lgbm - pred.true_rul)[late].mean():.1f} cycles")
print(f"Engines with true RUL <= 30 cycles: RMSE {np.sqrt(((pred.lgbm - pred.true_rul)[pred.true_rul <= 30]**2).mean()):.1f} (the region that triggers a hangar slot)")

LightGBM: 57% of test engines predicted late; mean lateness among them 11.9 cycles
Engines with true RUL <= 30 cycles: RMSE 5.6 (the region that triggers a hangar slot)


In [3]:
imp = pd.DataFrame(m['top_features'], columns=['feature', 'splits']).set_index('feature')
fig, ax = plt.subplots(figsize=(7, 4.5))
imp['splits'][::-1].plot.barh(ax=ax, color='#4E8C6A'); ax.set_title('LightGBM feature importance (split count), top 15'); ax.set_xlabel('splits')
fig.tight_layout(); fig.savefig('../docs/figures/rul_feature_importance.png'); plt.close(fig)
imp.head(15)

,splits
feature,
s13_std,2343
s11_std,2304
s8_std,2293
s14_std,2271
s7_std,2228
s12_std,2228
s4_std,2179
s9_std,2175
s17_std,2132


## Reading the numbers

* LightGBM beats the GRU on both metrics and trains in seconds; the gap on the NASA score is larger than on RMSE because the GRU's errors skew late.
* Both learned models cut the constant baseline's score by two orders of magnitude, which is the point of the capped-label formulation.
* Rolling-standard-deviation features dominate the importances: as an engine degrades its sensors get noisier before they drift, and the 30-cycle window encodes that. Sensors 13, 11, 8, 14 and 7 (core speeds, fan speed, corrected speeds) lead, which an engineer can interrogate directly.
* C-MAPSS is simulated. It proves the method, not the numbers: real engine data has sensor dropouts, maintenance resets and censored histories that this set does not.